# Etapa 6. Classificação do ENOS pelo ONI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/spi_oni_pr/blob/main/notebooks/06_oni_enos.ipynb)

O **ONI** (Oceanic Niño Index) é a anomalia da temperatura do mar na região Niño 3.4 do Pacífico, em média móvel de
3 meses, calculada pela NOAA. A regra para os eventos:

- **El Niño**: ONI ≥ +0,5 °C por pelo menos 5 trimestres móveis seguidos;
- **La Niña**: ONI ≤ −0,5 °C pelo mesmo critério;
- **Neutro**: os demais meses.

A intensidade vem do valor extremo do ONI no evento: fraca (0,5 a 0,9), moderada (1,0 a 1,4), forte (1,5 a 1,9) e
muito forte (≥ 2,0). As funções estão em `src/enos.py`; o capítulo 6 do caderno faz a regra à mão.

In [ ]:
# ==========================================================================================
# CÉLULA 0. PREPARAR O AMBIENTE (rode sempre primeiro, toda vez que abrir o notebook)
# ==========================================================================================
# Esta célula:
#   1) define DE ONDE os dados serão lidos (direto do GitHub, no Colab);
#   2) baixa as funções do projeto (pasta src/ do GitHub): fdp.py, spi.py, enos.py, estatistica.py;
#   3) conecta o seu Google Drive e usa a pasta spi_oni_pr/resultados, onde ficam os resultados
#      de uma etapa para a outra.
import sys
import urllib.request
from pathlib import Path

NO_COLAB = "google.colab" in sys.modules
REPO = "https://raw.githubusercontent.com/fcoliveira-utfpr/spi_oni_pr/main/"

if Path("../dados").exists():                   # computador do orientador, dentro da pasta do projeto
    DADOS = "../dados/"
    sys.path.append("../src")
else:                                           # Colab
    DADOS = REPO + "dados/"
    Path("src").mkdir(exist_ok=True)
    for arquivo in ["fdp.py", "spi.py", "enos.py", "estatistica.py"]:
        urllib.request.urlretrieve(REPO + "src/" + arquivo, "src/" + arquivo)   # baixa o arquivo
    sys.path.append("src")

if NO_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    SAIDA = Path("/content/drive/MyDrive/spi_oni_pr/resultados")
else:
    SAIDA = Path("../resultados")
SAIDA.mkdir(parents=True, exist_ok=True)
print("Dados lidos de:", DADOS)
print("Resultados gravados em:", SAIDA)

In [ ]:
import numpy as np
import pandas as pd
import enos

# Aparência padrão dos gráficos do trabalho (rode uma vez; vale para todos os gráficos do notebook)
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 10,
    "axes.spines.top": False,       # tira a borda de cima
    "axes.spines.right": False,     # tira a borda da direita
    "axes.grid": True,              # grade de fundo...
    "grid.color": "#e6e5e0",        # ...bem clara, para não competir com os dados
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,         # grade atrás das barras e linhas
})

AZUL = "#2a78d6"                    # cor única quando há só uma série
# uma cor fixa para cada estação em destaque (a cor acompanha a estação em todas as figuras)
CORES = {"Santa Helena": "#2a78d6", "Cascavel": "#eb6834", "Foz do Iguaçu": "#1baf7a", "Mercedes": "#eda100"}

from matplotlib.colors import LinearSegmentedColormap

# escala divergente: vermelho (seco / negativo) - cinza (zero) - azul (úmido / positivo)
DIVERGENTE = LinearSegmentedColormap.from_list("div", ["#a3201f", "#e34948", "#f0efec", "#2a78d6", "#184f95"])
# cores das fases do ENOS: El Niño = águas quentes (vermelho), La Niña = águas frias (azul)
CORES_FASE = {"El Niño": "#e34948", "Neutro": "#9a9890", "La Niña": "#2a78d6"}

## 1. O ONI

Cada trimestre móvel (DJF = dezembro-janeiro-fevereiro, JFM, ...) é atribuído ao seu **mês central** (DJF → janeiro).

In [ ]:
oni = pd.read_csv(DADOS + "oni.csv")
tab = enos.oni_mensal(oni)
tab.tail()

## 2. Fases e eventos

A classificação é feita na série inteira (desde 1950), para não cortar no meio um evento que começou antes de 1981.
Depois, ficamos com o período do estudo.

In [ ]:
tab = enos.classificar(tab)
estudo = tab[(tab["ano"] >= 1981) & ((tab["ano"] < 2020) | ((tab["ano"] == 2020) & (tab["mes"] <= 7)))]
estudo[["ano", "mes", "oni", "fase", "intensidade"]].to_csv(SAIDA / "enos_mensal.csv", index=False)

print("Meses no período:", len(estudo))
print((estudo["fase"].value_counts(normalize=True) * 100).round(1))

A lista de eventos que tocam o período de 1981 a 2020 (Tabela 6 do TCC). **Confira** com a tabela oficial da NOAA:
https://www.cpc.ncep.noaa.gov/products/analysis_monitoring/ensostuff/ONI_v5.php

In [ ]:
eventos_estudo = sorted(estudo.loc[estudo["evento"] > 0, "evento"].unique())
tabela6 = enos.lista_eventos(tab[tab["evento"].isin(eventos_estudo)])
tabela6.to_csv(SAIDA / "tabela06_eventos_enos.csv", index=False)
tabela6

In [ ]:
print(tabela6.groupby(["fase", "intensidade"]).size())

## 3. O ONI no tempo (Figura 7 do TCC)

Barras coloridas pela fase do mês: vermelho = El Niño, azul = La Niña, cinza = neutro.

In [ ]:
datas = pd.to_datetime(dict(year=estudo["ano"], month=estudo["mes"], day=1))
cores = [CORES_FASE[f] for f in estudo["fase"]]

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.bar(datas, estudo["oni"], width=31, color=cores)
ax.axhline(0.5, color="#9a9890", linewidth=0.8, linestyle="--")
ax.axhline(-0.5, color="#9a9890", linewidth=0.8, linestyle="--")
ax.set_ylabel("ONI (°C)")
ax.set_title("Oceanic Niño Index (NOAA/CPC), 1981–2020")
# legenda feita à mão: um quadradinho de cada cor
for fase, cor in CORES_FASE.items():
    ax.bar([], [], color=cor, label=fase)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False, fontsize=8)
fig.savefig(SAIDA / "fig07_oni_fases.png", dpi=300, bbox_inches="tight")
plt.show()

---

## O que observar

Devem aparecer, entre outros, os El Niños muito fortes de 1982/83, 1997/98 e 2015/16 e as La Niñas fortes de
1988/89, 1998–2000, 2007/08 e 2010/11. Pequenas diferenças de início e fim em relação à tabela da NOAA podem
ocorrer: a NOAA publica o ONI com uma casa decimal; o arquivo tem duas.

## Exercícios

**Exercício 1.** Quantos meses do período foram de El Niño, de La Niña e neutros? Qual fase foi mais comum?

In [ ]:
# escreva sua resposta aqui

**Exercício 2.** Em 2020 começou uma La Niña que durou até 2023 (a "La Niña tripla"). A série de chuva do trabalho
termina em julho de 2020. Por que isso é uma limitação para as conclusões? Escreva a resposta pensando no item
"Considerações finais" das Conclusões.

*Escreva sua resposta aqui.*

---

**Entregar.** `enos_mensal.csv` (no Drive), `fig07`, `tabela06` e os exercícios. **O que escrever:** item 5.5 do
TCC2 e conferir o item 4.5.